In [4]:
pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.1 MB/s eta 0:00:00


In [5]:
pip install transformers accelerate huggingface_hub

In [8]:
# Libraries
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from transformers import AutoTokenizer

# Integrate youtube_api.py code:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# Load tokenizer with left padding for Qwen
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side ='left')


# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    tokenizer = tokenizer,
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200 # increase to prevent incomplete output
)

import json

with open("video_claims_new.json", "r") as f:
    data = json.load(f)

all_claims = " \n".join(
    claim for item in data for claim in item["claims"]
)

print(all_claims)

def chunk_text(text, chunk_size=800, overlap=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # overlap keeps context between chunks
    return chunks



def extract_after_keyword(text, keywords):
    """Try multiple keyword markers, return text after the last one found."""
    if isinstance(keywords, str):
        keywords = [keywords]

    best_idx = -1
    best_keyword = None
    for keyword in keywords:
        idx = text.rfind(keyword)
        if idx > best_idx:
            best_idx = idx
            best_keyword = keyword

    if best_idx != -1:
        return text[best_idx + len(best_keyword):].strip()
    return text.strip()

def process_claims_trends_chunk(chunk, pipe):
    prompt = (
        "You are helping YouTube gamers improve their content.\n\n"
        "Given the factual claims below, identify upcoming trends in gaming videos.\n"
        "Rules:\n"
        "- Output bullet points ONLY, starting each line with '-'\n"
        "- Be concise\n"
        "- Merge similar ideas\n"
        "- Do NOT repeat the prompt or claims\n\n"
        f"Claims:\n{chunk}\n\nTrends:"
    )
    output = pipe(prompt, max_new_tokens=300)
    raw = output[0]["generated_text"]
    return extract_after_keyword(raw, "Trends:")

def process_claims_chunk(chunk, pipe):
    prompt = (
        "You are helping YouTube gamers improve their content.\n\n"
        "Given the factual claims below, convert them into actionable recommendations.\n"
        "Rules:\n"
        "- Output bullet points ONLY, starting each line with '-'\n"
        "- Be concise\n"
        "- Merge similar ideas\n"
        "- Focus on advice (not raw claims)\n"
        "- Give creators advice on what type of games they should play\n"
        "- Do NOT repeat the prompt or claims\n\n"
        f"Claims:\n{chunk}\n\nRecommendations:"
    )
    output = pipe(prompt, max_new_tokens=300)
    raw = output[0]["generated_text"]
    return extract_after_keyword(raw, "Recommendations:")

def deduplicate_recommendations(text, pipe):
    prompt = (
        "Clean and deduplicate these points.\n"
        "Merge similar points and keep the clearest version.\n"
        "Output bullet points ONLY, starting each line with '-'.\n"
        "Do NOT repeat these instructions in your output.\n\n"
        f"{text}\n\nCleaned:"
    )
    output = pipe(prompt, max_new_tokens=500)
    raw = output[0]["generated_text"]
    return extract_after_keyword(raw, ["Deduplicated:", "Cleaned:", "Reformatted:"])


def claim_extraction(text, pipe):
    chunks = chunk_text(text, chunk_size=800, overlap=100)
    all_recommendations = []
    all_trends = []

    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}/{len(chunks)}...", end=" ", flush=True)
        all_recommendations.append(process_claims_chunk(chunk, pipe))
        all_trends.append(process_claims_trends_chunk(chunk, pipe))
        print("done")

    combined_recommendations = "\n".join(all_recommendations)
    combined_trends = "\n".join(all_trends)
    """if len(chunks) > 1:
        print("  Deduplicating...")
        combined_recommendations = deduplicate_recommendations(combined_recommendations, pipe)
        combined_trends = deduplicate_recommendations(combined_trends, pipe)"""

    output = []

    for line in combined_recommendations.splitlines():
        line = line.strip().lstrip("-•*123456789. ")
        if line:
            output.append({"type": "recommendation", "content": line})

    for line in combined_trends.splitlines():
        line = line.strip().lstrip("-•*123456789. ")
        if line:
            output.append({"type": "trend", "content": line})

    return output

# Save to JSON file (one entry per line)
results = claim_extraction(all_claims, pipe)

with open("output.json", "w") as f:
    for entry in results:
        f.write(json.dumps(entry) + "\n")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The objective has been compromised and an emergency fail-safe is being initiated. 
The main point of the text is that the narrator is playing a game called "Troll Dog Tower" with his friend Kaylus, where he has to control a dog character while his friend controls the human character. The game involves completing obstacles together, and the narrator jokes about Kaylus being unaware of their roles. 
The text describes a video game scenario involving characters controlling multiple dogs through a magical carpet ride, with various interactions and challenges as players attempt to navigate obstacles and avoid falling off the carpet. 
The player is attempting to defeat an obstacle involving multiple dogs while being chased by someone else's dogs. 
The user is playing a game where they must catch a "dog" by throwing it back and forth across a room. 
The text discusses the upcoming NFC Championship game and the importance of Micah Parsons joining the team to provide an extra boost in prestige.

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
  Chunk 2/2... 

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done
